# 충남대학교 캠퍼스 챗봇 — Colab 전체 실행 (COLAB_RUN_ALL)

이 노트북 하나로 학습 → 인덱스 빌드 → 산출물 생성 → 커스텀 UI 공개링크까지 한 번에 됩니다.
위에서부터 셀을 순서대로 하나씩 실행하세요. (멈추지 말고 위→아래)

## 시작 전 꼭 확인
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → 하드웨어 가속기 = T4 GPU 로 설정하고 저장.
2. 그다음 셀을 위에서부터 차례대로 실행.

## 시간 안내 (T4 기준)
- 분류기 학습: 약 3~6분
- 인덱스(chroma) 빌드: 약 15분
- Task2 첫 실행: EXAONE 모델 다운로드 약 2분 + 추론

## 채점과의 관계
실제 채점은 classifier.ipynb 와 chatbot.sh 만 실행합니다.
이 노트북은 그걸 포함해 학습/검증/데모까지 한 번에 돌려보는 올인원 노트북입니다.
즉 이 노트북이 끝까지 잘 돌면, 채점 경로(classifier.ipynb · chatbot.sh)도 잘 됩니다.

## 1) 런타임 GPU 확인

T4 GPU가 잡혔는지 먼저 확인합니다. 아래에서 Tesla T4 와 CUDA available: True 가 보여야 정상입니다.
만약 GPU가 안 잡히면 런타임 → 런타임 유형 변경에서 T4 GPU로 바꾸고 다시 실행하세요.

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2) 코드 받기 + 의존성 설치

레포를 clone 하고 그 폴더로 들어간 뒤 requirements.txt 를 설치합니다.
설치는 몇 분 걸립니다. (torch 2.5.1 / transformers<4.49 / sentence-transformers 3.4.1 / bitsandbytes 등)
이미 cnu-llm-bot 폴더가 있어 clone 이 실패해도, 바로 아래 %cd 와 설치만 다시 돌리면 됩니다.

In [ ]:
!git clone https://github.com/Longarden/cnu-llm-bot.git
%cd cnu-llm-bot
!pip install -q -r requirements.txt
print("[ok] 설치 완료. 현재 폴더가 cnu-llm-bot 인지 위 %cd 출력으로 확인하세요.")

## 3) 분류기 학습 (Task1 모델 만들기) — 택1

아래 둘 중 하나만 하면 됩니다.

- (A) 처음부터 학습: roberta-large 로 새로 학습 (T4에서 약 3~6분). 아래 학습 셀 실행.
- (B) 드라이브 백업 복원: 이전에 model 을 드라이브에 백업해뒀다면 학습을 건너뛰고 복원만.

주의(중요): 이 프로젝트는 valid = test 로 같은 셋을 씁니다. 그래서 학습 로그에 찍히는
[eval] f1_macro 는 실제보다 낙관적(높게 나오는) 값입니다. 채점 일반화 성능과는 다를 수 있습니다.

### (A) 처음부터 학습 — roberta-large
학습이 끝나면 model/ 폴더에 분류기가 저장되고, 로그 맨 끝에 [eval] f1_macro 가 출력됩니다.

In [ ]:
# (A) 분류기 학습: klue/roberta-large, max_len 128, 6 epochs → model/ 에 저장
!CLS_MODEL=klue/roberta-large CLS_MAX_LEN=128 CLS_EPOCHS=6 python scripts/train_classifier.py

### (B) 드라이브 백업 복원 (학습 건너뛸 때만)
이전에 model 을 /content/drive/MyDrive/cnu_model 로 백업해뒀다면 아래를 실행하세요.
백업이 없으면 이 셀은 건너뛰고 위 (A) 학습을 쓰세요.

In [ ]:
# (B) 드라이브에서 분류기 복원 (학습 대신)
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/cnu_model /content/cnu-llm-bot/model
print("[ok] model/ 복원 완료")

### 분류기 config 확인
학습/복원된 분류기가 제대로인지 model/config.json 을 확인합니다.
roberta-large 면 _name_or_path 에 klue/roberta-large, hidden_size 가 1024 로 보여야 합니다.

In [ ]:
import json
cfg = json.load(open('model/config.json', encoding='utf-8'))
print("architectures :", cfg.get('architectures'))
print("_name_or_path :", cfg.get('_name_or_path'))
print("hidden_size   :", cfg.get('hidden_size'))
print("num_labels    :", cfg.get('num_labels', len(cfg.get('id2label', {})) or '?'))

## 4) 인덱스(벡터DB) 빌드 — 택1

RAG 검색용 chroma 벡터DB를 만듭니다. 둘 중 하나만 하면 됩니다.

- (A) 빌드: bge-m3 임베딩으로 새로 빌드 (T4에서 약 15분). 아래 빌드 셀 실행.
- (B) 드라이브 복원: 이전에 chroma_db 를 백업해뒀다면 복원만 (15분 절약).

### (A) 인덱스 새로 빌드 (약 15분)

In [ ]:
# (A) chroma 벡터DB 빌드 (bge-m3)
!python scripts/rebuild_index.py

### (B) 드라이브 백업 복원 (빌드 건너뛸 때만)

In [ ]:
# (B) 드라이브에서 chroma_db 복원 (빌드 대신)
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/cnu_chroma_db /content/cnu-llm-bot/chroma_db
print("[ok] chroma_db 복원 완료")

## 5) Task1 산출물 — 질문유형 분류 (cls_output.json)

분류기로 테스트 질문들을 0~4로 분류해 outputs/cls_output.json 을 만듭니다.
바로 아래에서 포맷(리스트 / id·question·label 0~4 정수)을 자동 검증하고 라벨 분포를 출력합니다.

In [ ]:
!python src/classifier.py

In [ ]:
import json
from collections import Counter

rows = json.load(open('outputs/cls_output.json', encoding='utf-8'))
assert isinstance(rows, list) and len(rows) > 0, "cls_output.json 이 비어있거나 리스트가 아닙니다."
for i, r in enumerate(rows):
    assert 'id' in r,       f"{i}번째 항목에 id 가 없습니다."
    assert 'question' in r, f"{i}번째 항목에 question 이 없습니다."
    assert 'label' in r,    f"{i}번째 항목에 label 이 없습니다."
    lab = r['label']
    assert isinstance(lab, int) and 0 <= lab <= 4, \
        f"{i}번째 label 이 0~4 정수가 아닙니다: {lab!r}"
print(f"[OK] Task1 포맷 통과 — 총 {len(rows)}건")
print("라벨 분포(0 졸업요건 / 1 학교공지 / 2 학사일정 / 3 식단 / 4 통학셔틀):")
for lab, cnt in sorted(Counter(r['label'] for r in rows).items()):
    print(f"  label {lab}: {cnt}건")

## 6) Task2 산출물 — 챗봇 답변 (chat_output.json)

각 질문을 chat_answer(분류 → RAG/라이브 → EXAONE 생성)로 답해 outputs/chat_output.json 을 만듭니다.
EXAONE 모델 첫 다운로드로 처음엔 시간이 좀 걸립니다.
아래 검증 셀은 포맷(리스트 / id·user·model)과, model 칸이 비었거나 "생성 오류"가 아님을 확인합니다.

In [ ]:
!python src/gen_chat_output.py

In [ ]:
import json

rows = json.load(open('outputs/chat_output.json', encoding='utf-8'))
assert isinstance(rows, list) and len(rows) > 0, "chat_output.json 이 비어있거나 리스트가 아닙니다."
bad = []
for i, r in enumerate(rows):
    assert 'id' in r,    f"{i}번째 항목에 id 가 없습니다."
    assert 'user' in r,  f"{i}번째 항목에 user 가 없습니다."
    assert 'model' in r, f"{i}번째 항목에 model 이 없습니다."
    m = (r['model'] or "").strip()
    if (not m) or m.startswith("생성 오류"):
        bad.append((i, r['user'][:30], m[:40]))
assert not bad, "빈 답변/생성 오류가 있습니다 → " + "; ".join(
    f"[{i}] {u} :: {m}" for i, u, m in bad)
print(f"[OK] Task2 포맷 통과 — 총 {len(rows)}건, 모든 답변이 채워졌습니다.")
print("예시 답변 2건:")
for r in rows[:2]:
    print("  Q:", r['user'][:50])
    print("  A:", (r['model'] or "")[:120], "...")

## 7) 새 휘발성 라우팅 점검 (live vs static)

chat_answer 가 "변하는 정보"는 라이브 크롤(live)로, "안 변하는 정보"는 정적 RAG(static)로
가는지 확인합니다. return_meta=True 의 source 값으로 판정합니다.
기대: "오늘 학식" · "다음주 셔틀" → live, "셔틀 노선" · "졸업 요건" → static.

In [ ]:
import os
os.environ['CHAT_REALTIME'] = '1'
from src.chat_pipeline import chat_answer

probes = [
    ("오늘 학식 뭐 나와요?",        "live"),
    ("다음주 셔틀 운행하나요?",      "live"),
    ("셔틀 노선 알려줘",            "static"),
    ("졸업 요건 알려줘",            "static"),
]
print(f"{'질문':<22} {'기대':<8} {'실제':<8} 판정")
print("-" * 52)
for q, expect in probes:
    _, meta = chat_answer(q, return_meta=True)
    src = meta.get('source', '?')
    mark = "OK" if src == expect else "확인필요"
    print(f"{q:<22} {expect:<8} {src:<8} {mark}")
print("\n(네트워크 사정으로 라이브 크롤이 실패하면 static 폴백이 될 수 있습니다 — 정상 동작입니다.)")

## 8) Task3 산출물 — 실시간 반영 (realtime_output.json)

셔틀/식단/공지를 라이브 크롤해 최신 정보로 답한 outputs/realtime_output.json 을 만듭니다.
검증 셀은 포맷(리스트 / id·user·model)과, model 칸이 비었거나 "생성 오류"가 아님을 확인합니다.

In [ ]:
!python src/realtime_model.py

In [ ]:
import json

rows = json.load(open('outputs/realtime_output.json', encoding='utf-8'))
assert isinstance(rows, list) and len(rows) > 0, "realtime_output.json 이 비어있거나 리스트가 아닙니다."
bad = []
for i, r in enumerate(rows):
    assert 'id' in r,    f"{i}번째 항목에 id 가 없습니다."
    assert 'user' in r,  f"{i}번째 항목에 user 가 없습니다."
    assert 'model' in r, f"{i}번째 항목에 model 이 없습니다."
    m = (r['model'] or "").strip()
    if (not m) or m.startswith("생성 오류"):
        bad.append((i, r['user'][:30], m[:40]))
assert not bad, "빈 답변/생성 오류가 있습니다 → " + "; ".join(
    f"[{i}] {u} :: {m}" for i, u, m in bad)
print(f"[OK] Task3 포맷 통과 — 총 {len(rows)}건, 모든 답변이 채워졌습니다.")
for r in rows[:2]:
    print("  Q:", r['user'][:50])
    print("  A:", (r['model'] or "")[:120], "...")

## 9) UI 미리보기 (빠른 점검, 모델 없이)

UI_MOCK=1 로 무거운 모델/검색 없이 UI 화면만 빠르게 띄워봅니다.
이 셀은 블로킹입니다(계속 떠 있는 게 정상). 출력에 trycloudflare.com 공개링크가 뜨면 클릭해서
화면이 잘 나오는지만 확인하세요. 확인 후에는 셀 왼쪽 정지(■) 버튼으로 멈추고 다음 셀로 갑니다.
(공개링크 발급을 위해 GRADIO_SHARE=1 을 함께 줍니다.)

In [ ]:
# UI만 미리보기 (모델 미로딩). 공개링크 확인 후 정지(■) 버튼으로 멈추세요.
!UI_MOCK=1 GRADIO_SHARE=1 python src/chatbot_ui.py

## 10) 실제 UI 데모 — 공개링크 (영상 촬영용)

채점 경로와 동일한 chatbot.sh 를 실행합니다.
순서: (1) chat_output 생성 → (2) realtime_output 생성 → (3) 커스텀 UI 띄움.
UI는 src/chatbot_ui.py 가 cloudflared 퀵터널로 공개링크(https://...trycloudflare.com)를 발급합니다.

이 셀은 블로킹입니다(계속 떠 있는 게 정상). 출력에서 공개링크가 뜨면 클릭해 접속하고,
그 상태로 2분 시연 영상을 촬영하세요. 촬영이 끝나면 정지(■) 버튼으로 멈춥니다.

아래 첫 줄에서 UI에 필요한 fastapi/uvicorn/starlette 를 안전하게 한 번 더 설치합니다(이미 있으면 무해).

In [ ]:
!pip install -q fastapi uvicorn starlette
# (1) chat_output → (2) realtime_output → (3) 커스텀 UI + trycloudflare 공개링크. 블로킹(2분 영상).
!bash chatbot.sh

## 11) 제출 패키징 (옵션)

제출용 zip 을 만듭니다. INCLUDE_MODEL=1 이면 분류기 가중치까지 동봉합니다(용량 큼).
NAME 에 본인 이름을 넣으세요(예: 장정원). 결과는 dist/Termproject_<이름>.zip 에 생깁니다.
zip 이 너무 크면(드라이브 업로드 제한) 가중치는 빼고(INCLUDE_MODEL 제거) 드라이브로 따로 올리세요.

In [ ]:
!INCLUDE_MODEL=1 NAME=장정원 bash scripts/package_submission.sh
!ls -lh dist/Termproject_장정원.zip

In [ ]:
# (옵션) zip 을 드라이브로 백업
from google.colab import drive
drive.mount('/content/drive')
!cp dist/Termproject_장정원.zip /content/drive/MyDrive/
print("[ok] 드라이브에 zip 백업 완료")

## 12) 제출 전 체크리스트

- [ ] 제출 폴더 이름이 Termproject_<본인이름> 형식인가 (예: Termproject_장정원)
- [ ] 채점 경로 점검: src/classifier.ipynb 와 chatbot.sh 가 단독으로 잘 돌아가는가
- [ ] outputs/ 에 cls_output.json, chat_output.json, realtime_output.json 3개가 다 있는가
- [ ] 각 산출물 포맷 통과(위 검증 셀들이 모두 [OK])했는가 — 빈 답변/"생성 오류" 없음
- [ ] requirements.txt 가 함께 들어갔는가 (조교가 pip install -r 로 환경 재현)
- [ ] model/ 가중치 또는 복원 안내(DOWNLOAD_MODEL.txt)가 포함됐는가
- [ ] UI 시연 영상 2분 (chatbot.sh 공개링크에서 질문/답변/대화흐름 보여주기)
- [ ] 발표 자료/대본 5분 분량 준비